# 02 - Prétraitement des données SIEVE

## Objectif

Préparer les six variantes du dataset SIEVE pour l'entraînement
des modèles de classification.

## Étapes

1. Charger les données.
2. Vérifier et nettoyer les valeurs manquantes.
3. Supprimer les doublons exacts, si cela est justifié.
4. Nettoyer les textes sans supprimer les informations importantes.
5. Encoder les catégories.
6. Séparer les données en ensembles d'entraînement, de validation et de test.
7. Sauvegarder les données préparées.

In [1]:
import pandas as pd
import numpy as np
import os
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


In [3]:
import os
from data_loader import load_log

datasets = {}

for file in files:
    path = os.path.join(DATA_DIR, file)
    df = load_log(path)

    datasets[file] = df
    print(f"{file} : {df.shape}")

output_0.log : (35000, 2)
output_0.1.log : (35000, 2)
output_0.2.log : (35000, 2)
output_0.3.log : (35000, 2)
output_0.4.log : (35000, 2)
output_0.5.log : (35000, 2)


In [8]:
def clean_log(text):

    text = str(text)

    # Supprimer les espaces au début et à la fin
    text = text.strip()

    # Remplacer plusieurs espaces par un seul
    text = re.sub(r"\s+", " ", text)

    return text

def clean_log(text):

    text = str(text)

    # Supprimer les espaces au début et à la fin
    text = text.strip()

    # Remplacer plusieurs espaces par un seul
    text = re.sub(r"\s+", " ", text)

    return text

display(
    datasets["output_0.log"][["category", "log"]].head(5)
)

,category,log
0,authentication-failed,[Wed Nov 03 19:30:11 1976] [error] [client 217...
1,authentication-failed,[Sat May 21 08:27:17 1988] [error] [client 195...
2,authentication-failed,[Sat Jul 21 16:54:26 2001] [error] [client 129...
3,authentication-failed,[Thu Apr 15 21:41:51 1971] [error] [client 199...
4,authentication-failed,[Sun May 22 01:51:12 1977] [error] [client 122...


In [9]:
# Récupérer toutes les catégories des six variantes
all_categories = pd.concat(
    [df["category"] for df in datasets.values()],
    ignore_index=True
)

# Créer un encodeur commun
label_encoder = LabelEncoder()

label_encoder.fit(all_categories)

# Encoder les catégories dans chaque fichier
for name in datasets:

    datasets[name]["label"] = label_encoder.transform(
        datasets[name]["category"]
    )

print("Encodage terminé.")
print("Nombre de classes :", len(label_encoder.classes_))

label_mapping = pd.DataFrame({
    "category": label_encoder.classes_,
    "label": range(len(label_encoder.classes_))
})

display(label_mapping)

Encodage terminé.
Nombre de classes : 30


,category,label
0,authentication-failed,0
1,authentication-success,1
2,connection-closed,2
3,connection-failed,3
4,connection-opened,4
5,database-operation,5
6,directory-changed,6
7,directory-created,7
8,directory-deleted,8
9,file-action-failure,9


In [10]:
splits = {}

for name, df in datasets.items():

    X = df["log"]
    y = df["label"]

    # 70 % entraînement et 30 % temporaire
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=42,
        stratify=y
    )

    # Diviser les 30 % restants en validation et test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp
    )

    splits[name] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test
    }

    print("\n", name)
    print("Train :", len(X_train))
    print("Validation :", len(X_val))
    print("Test :", len(X_test))


 output_0.log
Train : 24500
Validation : 5250
Test : 5250

 output_0.1.log
Train : 24500
Validation : 5250
Test : 5250

 output_0.2.log
Train : 24500
Validation : 5250
Test : 5250

 output_0.3.log
Train : 24500
Validation : 5250
Test : 5250

 output_0.4.log
Train : 24500
Validation : 5250
Test : 5250

 output_0.5.log
Train : 24500
Validation : 5250
Test : 5250


In [11]:
OUTPUT_DIR = "../processed_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for name, data in splits.items():

    variant = name.replace(".log", "")

    for split_name, values in data.items():

        output = pd.DataFrame({
            "log": values.values if split_name.startswith("X_")
                   else splits[name][split_name.replace("y_", "X_")].values,
            "label": values.values if split_name.startswith("y_")
                     else splits[name][split_name.replace("X_", "y_")].values
        })

        output.to_csv(
            os.path.join(
                OUTPUT_DIR,
                f"{variant}_{split_name}.csv"
            ),
            index=False
        )

print("Sauvegarde terminée.")

Sauvegarde terminée.


In [12]:
for name, data in splits.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    for split_name, values in data.items():

        print(
            split_name,
            ":",
            len(values),
            "éléments"
        )


output_0.log
X_train : 24500 éléments
X_val : 5250 éléments
X_test : 5250 éléments
y_train : 24500 éléments
y_val : 5250 éléments
y_test : 5250 éléments

output_0.1.log
X_train : 24500 éléments
X_val : 5250 éléments
X_test : 5250 éléments
y_train : 24500 éléments
y_val : 5250 éléments
y_test : 5250 éléments

output_0.2.log
X_train : 24500 éléments
X_val : 5250 éléments
X_test : 5250 éléments
y_train : 24500 éléments
y_val : 5250 éléments
y_test : 5250 éléments

output_0.3.log
X_train : 24500 éléments
X_val : 5250 éléments
X_test : 5250 éléments
y_train : 24500 éléments
y_val : 5250 éléments
y_test : 5250 éléments

output_0.4.log
X_train : 24500 éléments
X_val : 5250 éléments
X_test : 5250 éléments
y_train : 24500 éléments
y_val : 5250 éléments
y_test : 5250 éléments

output_0.5.log
X_train : 24500 éléments
X_val : 5250 éléments
X_test : 5250 éléments
y_train : 24500 éléments
y_val : 5250 éléments
y_test : 5250 éléments
